In [ ]:
import json
import os
import stat
import sys
from pathlib import Path

# 项目含 03_results/run_manifest.json；Pipeline 为 Alopex 安装目录。
# 建议填写绝对路径字符串；None 使用同名环境变量，再按项目 manifest 定位。
DNA_PROJECT_ROOT = None    # 例如 "/absolute/path/to/Patient001"
DNA_PIPELINE_ROOT = None   # 例如 "/absolute/path/to/Alopex"
TSS_BED = None             # 默认使用项目物种的 TSS；可填外部 BED 绝对路径
SKIP_TSS = False           # 无需 TSS 时设 True，不能同时指定 TSS_BED

# 导入 Alopex 前，也要区分真正缺席与悬空链接、访问失败。
def _stat_if_present(path):
    probe = path
    while True:
        try:
            probe.lstat()
        except FileNotFoundError:
            if probe == probe.parent:
                return None
            probe = probe.parent
        else:
            state = probe.stat()
            return state if probe == path else None

# 导入前只定位源码；完整交付与身份校验由 qc.resolve_notebook_paths 负责。
_pipeline = DNA_PIPELINE_ROOT or os.environ.get("DNA_PIPELINE_ROOT")
if not _pipeline:
    _project = DNA_PROJECT_ROOT or os.environ.get("DNA_PROJECT_ROOT")
    _start = Path(_project or Path.cwd()).expanduser().resolve()
    _bases = [_start] if _project else [_start, *_start.parents]
    _manifest = None
    for _base in _bases:
        _candidate = _base / "03_results/run_manifest.json"
        _state = _stat_if_present(_candidate)
        if _state is None:
            if _stat_if_present(_base / "00_config/config.yaml") is not None:
                raise FileNotFoundError(f"Alopex project has no sealed delivery: {_candidate}")
            continue
        if not stat.S_ISREG(_state.st_mode):
            raise ValueError(f"Delivery manifest must be a regular file: {_candidate}")
        _manifest = _candidate
        break
    if _manifest is None:
        raise FileNotFoundError("找不到项目 run_manifest.json；请设置 DNA_PROJECT_ROOT。")
    _metadata = json.loads(_manifest.read_text(encoding="utf-8"))
    _pipeline = _metadata.get("pipeline") if isinstance(_metadata, dict) else None
    _pipeline = _pipeline.get("source_root") if isinstance(_pipeline, dict) else None
    if not isinstance(_pipeline, str) or not _pipeline.strip():
        raise FileNotFoundError(f"run_manifest.pipeline.source_root is missing: {_manifest}")
    _owner = _metadata.get("project_dir") if isinstance(_metadata, dict) else None
    if not isinstance(_owner, str) or not Path(_owner).is_absolute() or Path(_owner).resolve() != _base:
        raise ValueError(f"Run manifest belongs to a different project: {_manifest}")

_pipeline = Path(_pipeline).expanduser().resolve()
if not all((_pipeline / p).is_file() for p in
           ("core/alopex/downstream/qc.py", "core/run_pipeline.sh")):
    raise FileNotFoundError(f"Not a usable Alopex: {_pipeline}")
if str(_pipeline / "core") not in sys.path:
    sys.path.insert(0, str(_pipeline / "core"))
from alopex.downstream import qc
if Path(qc.__file__).resolve() != _pipeline / "core/alopex/downstream/qc.py":
    raise RuntimeError("A different Alopex is already loaded; restart the kernel before selecting another Pipeline.")

PATHS = qc.resolve_notebook_paths(DNA_PROJECT_ROOT, DNA_PIPELINE_ROOT)
from alopex.downstream import plotting

PROJECT_DIR, PIPELINE_PATH = PATHS.project_dir, PATHS.pipeline_root
print(f"Pipeline: {PIPELINE_PATH}\nProject: {PROJECT_DIR}\nQC results: {PATHS.qc_results_dir}")


In [ ]:
# 强制重算 / 重建开关；默认 False 时按内容与缓存身份自动判定复用
RECOMPUTE_CPG_STATS = False
RECOMPUTE_GINI = False
RECOMPUTE_TSS = False

# TSS 每个 CloneID 抽检 10%（向上取整）；不足 10 个全检，最多 100 个；1.0 为全量
TSS_SAMPLE_FRACTION = 0.10
TSS_SAMPLE_SEED = 0

# HQ 门槛随 QC 写入 SingleCpG；设置 None 时不启用 mapping 门槛。
MAPPING_THRESHOLD = None

qc.run_qc_processor(
    PATHS,
    tss_bed=TSS_BED,
    skip_tss=SKIP_TSS,
    tss_sample_fraction=TSS_SAMPLE_FRACTION,
    tss_sample_seed=TSS_SAMPLE_SEED,
    qc_filter_parameters={"mapping_threshold": MAPPING_THRESHOLD},
    recompute_cpg_stats=RECOMPUTE_CPG_STATS,
    recompute_gini=RECOMPUTE_GINI,
    recompute_tss=RECOMPUTE_TSS,
)

SINGLECPG_ADATA_PATH = PATHS.qc_results_dir / "SingleCpG_Adata.h5ad"
print(f"SingleCpG: {SINGLECPG_ADATA_PATH}")


In [ ]:
df_qc, clone_color_map = qc.load_notebook_qc_frame(PATHS)

PLOTS_DIR = PATHS.plots_dir


In [ ]:
df_qc

In [ ]:
df_qc.groupby("CloneID").median(numeric_only=True)

In [ ]:
df_qc.groupby("CloneID").mean(numeric_only=True)

In [ ]:
df_qc[['Sample_ID', 'HQ', 'QC_Failed_Criteria', 'QC_Filter_Reasons']]


In [ ]:
if not df_qc.empty:
    plotting.plot_wgbs_qc_panel(df_qc, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'WGBS_QC_Panel.pdf')


## 最终 QC 指标说明

| 模式 | 图的组织 | 使用场景 |
| --- | --- | --- |
| `per_clone_samples` | 每个指标生成一张图，每个 CloneID 一个分面；标注该组有效 N/输入 N 与中位数 | 检查每组内部的分布、低质量尾部及异常样本 |
| `distribution` | 每个指标生成一张图，在同一坐标轴按 CloneID 颜色叠加；标注全部所选组的有效 N/输入 N 与总体中位数（虚线） | 比较不同组的分布位置及离散程度；组数多时建议先选择部分组 |

两种模式均严格按 `FINAL_QC_METRIC` 选择指标：可以是一个字符串或有序列表，不会切换成固定的其他指标。支持 QC 表全部17个数值字段，包括 `Lambda_Meth_CpG_Rate%` 和 `pUC19_Meth_CpG_Rate%`（单位均为百分比，线性横轴）。未知指标或缺列直接报错；没有有限值时明确标注，不用0填补、不改画其他指标。

`FINAL_QC_CLONE_ID=None` 表示所有 CloneID；也可填写单个 CloneID 字符串或列表，例如 `['Clone-A', 'Clone-B']`（须替换为实际名称）。这是选组，不是按 Sample_ID 筛单个细胞。

两种模式共用100个分箱、配色与平滑样式。CpG 位点数正值使用对数横轴与对数等距分箱，含非正值时使用 signed-log1p 并保留真实零值。log/log1p 的主刻度定位于原始位点数的十进数量级，显示 `1E1、1E2、1E3…`；真实零值单独标为 `0`，log1p 近零区间与纯 log 的间距略有不同。宽范围适当跳过数量级以免标签重叠；窄范围没有整数数量级时显示实际中点的 E 记数值。不显示次刻度标签或轴外科学计数法偏移。甲基化率等其他指标为线性。纵轴为样本数 Counts，受各组样本量影响，不是概率密度；叠加组多时自动降低填充透明度以减少遮挡。平滑只用于展示，不改变 QC 或 HQ。横轴在变换空间两端各留4%空白；纵轴按全部已绘组的最高原始柱计数设范围，峰顶留20%、零基线下留3%空白，确保端点、峰顶和线宽完整落在图内。

每个指标单独显示并保存到 `PLOTS_DIR`：`Final_QC_Dashboard_<mode>_<metric>_<clone_selection>.pdf`，其中 `%` 写为 `pct`。两种模式文件名不同。更新绘图库后请重启 kernel，重跑首格及 QC 数据加载格，再执行本格；无需重跑 QC Processor 或生成矩阵。

- **Native_Mapping%**：backend 原生 mapping 率。BISCUIT 为 primary `MAPQ>=40` 的 individual reads，Bismark 为 unique concordant read pairs；两种口径单位不同，不能直接跨 backend 比较。
- **Duplicate_Pair_Rate%**：孔板路线为 duplicate pairs / backend accepted pairs；DD-MET5 使用建库 UMI，DD-MET3 使用插入序列片段标签，两者逐胞嘧啶做共识，此列为空，分子统计见上游 MultiQC 与 final manifest。
- **Final_Pair_Yield%**：最终保留的 read pairs / Cutadapt input pairs。

两个 backend 的 mapping policy 不同，默认不用统一的 mapping 阈值参与 HQ 判定；如项目已有经过验证的阈值，可显式设置 `MAPPING_THRESHOLD`。

In [ ]:
FINAL_QC_MODE = 'per_clone_samples' # 或 'distribution'，见上方两种模式说明

FINAL_QC_CLONE_ID = None # 全部 CloneID；或单个 CloneID 字符串/列表

FINAL_QC_METRIC = [
    'Sample_Unique_CpG_Sites',
    'Lambda_Meth_CpG_Rate%',
    'pUC19_Meth_CpG_Rate%',
    'Native_Mapping%',
    'Final_Pair_Yield%',
    'Duplicate_Pair_Rate%',
    'Non_CpG_Methylation%',
]

if not df_qc.empty:
    plotting.plot_final_qc_dashboard_batch(
        df_qc,
        metrics=FINAL_QC_METRIC,
        clone_id=FINAL_QC_CLONE_ID,
        clone_color_map=clone_color_map,
        out_dir=PLOTS_DIR,
        filename_prefix='Final_QC_Dashboard',
        mode=FINAL_QC_MODE,
    )


In [ ]:
PLATE_VIEW_METRIC = 'Native_Mapping%' #Sample_Unique_CpG_Sites

plotting.plot_plate_metric_map(
    df_qc,
    metric=PLATE_VIEW_METRIC,
    clone_color_map=clone_color_map,
    out_path=PLOTS_DIR / f'Plate_View_{PLATE_VIEW_METRIC}.pdf',
)

In [ ]:
SIGNAL_COMPOSITION_SIMPLIFY_SAMPLE_ID = True

if not df_qc.empty:
    plotting.plot_cpg_signal_composition(
        df_qc,
        out_path=PLOTS_DIR / 'CpG_Signal_Composition.pdf',
        simplify_sample_ids=SIGNAL_COMPOSITION_SIMPLIFY_SAMPLE_ID,
    )


In [ ]:
if not SKIP_TSS:
    df_tss = qc.load_notebook_tss_frame(PATHS)
    plotting.plot_tss_profile(df_tss, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'TSS_Profile.pdf')
else:
    print('本单元已按 SKIP_TSS 跳过 TSS 绘图。')


同时展示逐样本散点图（横轴：trimming 前 input read pairs；纵轴：最终宿主 unique CpG sites；按 CloneID 配色，不做深度归一化；每组拟合线及图例斜率复用原汇总的过原点拟合与异常点筛选）与原有斜率汇总，分别保存为 `CpG_Density_Unique_vs_Input_Pairs.pdf` 和 `CpG_Density.pdf`。

In [ ]:
if not df_qc.empty:
    df_density = qc.load_cpg_density_frame(
        PATHS,
    )
    plotting.plot_cpg_density(df_density, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'CpG_Density.pdf')


In [ ]:
if not df_qc.empty:
    plotting.plot_pca_batch_effect(df_qc, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'Batch_Effect_PCA.pdf')


## 逐染色体诊断
全部细胞的主核染色体及对照均保留；辅助contig保留实际观测。表中methyl/unmethyl/coverage为整数，无覆盖染色体的甲基化率为NaN。图按CloneID展示各染色体覆盖与甲基化中位数，原始逐细胞表保存在`Chromosome_QC_Information.csv`。


In [ ]:
df_chromosome = qc.load_chromosome_qc_frame(PATHS)
plotting.plot_chromosome_diagnostics(df_chromosome, clone_color_map=clone_color_map, out_path=PLOTS_DIR / 'Chromosome_QC.pdf')
df_chromosome.groupby(['CloneID', 'chrom'], sort=False, observed=True).median(numeric_only=True)
